# 4.2 backtrader 框架回测

## 学习目标
- 了解 backtrader 的核心架构（Strategy / Data Feed / Broker）
- 用 backtrader 实现 RSI 策略回测
- 利用 backtrader 绘图和绩效报告

In [ ]:
# 安装检查
try:
    import backtrader as bt
    print(f'backtrader 版本: {bt.__version__} ✅')
except ImportError:
    print('请先安装: pip install backtrader')

import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib
matplotlib.use('Agg')  # 非交互式后端（Jupyter 兼容）
import matplotlib.pyplot as plt

## 1. backtrader 核心架构

```
Cerebro（大脑）
  ├── DataFeed（数据喂入）— OHLCV 行情
  ├── Strategy（策略）— 信号 + 买卖逻辑
  ├── Broker（经纪商）— 资金、仓位、手续费
  └── Analyzer（分析器）— SharpeRatio, DrawDown 等
```

最核心的方法是 `Strategy.next()`：**每新增一根K线就会被调用一次**。

## 2. 准备数据

In [ ]:
# 下载数据并转换为 backtrader 格式
raw = yf.download('SPY', start='2018-01-01', end='2024-01-01', progress=False, auto_adjust=True)
raw = raw.droplevel(1, axis=1) if raw.columns.nlevels > 1 else raw

print(f'数据形状: {raw.shape}')
raw.tail(3)

## 3. 实现 RSI 策略

In [ ]:
class RSIStrategy(bt.Strategy):
    """
    RSI 均值回归策略：
    - RSI < 30 买入（超卖）
    - RSI > 70 卖出（超买）
    """
    params = (
        ('rsi_period', 14),
        ('rsi_overbought', 70),
        ('rsi_oversold', 30),
        ('stake_pct', 0.95),  # 每次用 95% 资金买入
    )

    def __init__(self):
        # 定义指标
        self.rsi = bt.indicators.RelativeStrengthIndex(
            self.data.close, period=self.params.rsi_period
        )
        self.order = None  # 追踪当前挂单

    def log(self, txt):
        dt = self.datas[0].datetime.date(0)
        print(f'[{dt}] {txt}')

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return
        if order.status == order.Completed:
            action = '买入' if order.isbuy() else '卖出'
            self.log(f'{action} @ {order.executed.price:.2f}, '
                     f'成本: {order.executed.value:.2f}, '
                     f'手续费: {order.executed.comm:.2f}')
        self.order = None

    def next(self):
        # 已有挂单则跳过
        if self.order:
            return

        if not self.position:  # 无持仓时
            if self.rsi < self.params.rsi_oversold:
                # 超卖 → 买入
                cash = self.broker.getcash()
                size = int(cash * self.params.stake_pct / self.data.close[0])
                self.order = self.buy(size=size)
        else:  # 有持仓时
            if self.rsi > self.params.rsi_overbought:
                # 超买 → 全部平仓
                self.order = self.sell(size=self.position.size)

print('RSIStrategy 定义完成 ✅')

## 4. 运行回测

In [ ]:
# 配置 Cerebro
cerebro = bt.Cerebro()
cerebro.addstrategy(RSIStrategy)

# 添加数据
data_feed = bt.feeds.PandasData(dataname=raw)
cerebro.adddata(data_feed)

# 经纪商设置
cerebro.broker.setcash(100_000)        # 初始资金 10 万
cerebro.broker.setcommission(0.001)   # 0.1% 手续费

# 添加分析器
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', riskfreerate=0.04)
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades')

# 运行
print(f'初始资金: {cerebro.broker.getvalue():,.2f}')
print('--- 交易记录 ---')
results = cerebro.run()
strat = results[0]
final_value = cerebro.broker.getvalue()
print('--- 回测完成 ---')
print(f'最终资产: {final_value:,.2f}')
print(f'总收益率: {(final_value / 100_000 - 1):.2%}')

## 5. 查看分析器结果

In [ ]:
sharpe  = strat.analyzers.sharpe.get_analysis()
drawdown = strat.analyzers.drawdown.get_analysis()
returns  = strat.analyzers.returns.get_analysis()
trades   = strat.analyzers.trades.get_analysis()

print('=' * 40)
print('       RSI 策略 — 回测结果')
print('=' * 40)
print(f'年化收益率  : {returns.get("rnorm100", 0):.2f}%')
print(f'夏普比率    : {sharpe["sharperatio"]:.2f}' if sharpe.get('sharperatio') else '夏普比率    : N/A')
print(f'最大回撤    : {drawdown.max.drawdown:.2f}%')
print(f'最大回撤天数: {drawdown.max.len} 天')

total_closed = trades.get('total', {}).get('closed', 0)
total_won = trades.get('won', {}).get('total', 0)
if total_closed > 0:
    print(f'总交易次数  : {total_closed}')
    print(f'胜率        : {total_won / total_closed:.2%}')
print('=' * 40)

## 6. 参数优化（网格搜索）

In [ ]:
# 对 RSI 超买超卖阈值做简单优化
results_grid = []

for oversold in [25, 30, 35]:
    for overbought in [65, 70, 75]:
        cerebro_opt = bt.Cerebro()
        cerebro_opt.addstrategy(RSIStrategy,
                                 rsi_oversold=oversold,
                                 rsi_overbought=overbought)
        cerebro_opt.adddata(bt.feeds.PandasData(dataname=raw.copy()))
        cerebro_opt.broker.setcash(100_000)
        cerebro_opt.broker.setcommission(0.001)
        cerebro_opt.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', riskfreerate=0.04)

        res = cerebro_opt.run()
        sr = res[0].analyzers.sharpe.get_analysis().get('sharperatio', None)
        final = cerebro_opt.broker.getvalue()
        results_grid.append({
            'oversold': oversold,
            'overbought': overbought,
            'final_value': round(final, 2),
            'total_return': f'{(final/100_000 - 1):.2%}',
            'sharpe': round(sr, 3) if sr else None
        })

opt_df = pd.DataFrame(results_grid).sort_values('final_value', ascending=False)
print('参数优化结果（按最终资产排序）：')
print(opt_df.to_string(index=False))

## 🎯 练习

1. 修改策略：当 RSI 从下穿越 30 后（而不是持续低于 30 时）才买入。
2. 加入止损：持仓亏损超过 5% 时强制平仓。
3. 使用 `cerebro.optstrategy()` 做完整网格搜索，找到最优参数组合。

---
**下一节** → `03_performance_metrics.ipynb`